# Harmonic game generator, clean
- players $i \in N$
- finite actions $a_i \in A_i$ for each player
- action profiles $A = \prod_{i \in N}A_i$ 
- payoff functions $u_i : A \to \mathbb{R}$
- standard game theoretic notation for $(a_i, a_{-i}) \in A$
- harmonic measure : $\mu_{i a_i} > 0$ for each $a_i \in A_i$ and $i \in N$

A game with fixed number of players and actions is harmonic if and only if there exist harmonic measures such that, for each action profile $a \in A$, the following conservation condition holds:

$$
\sum_{b_i \in A_i} \mu_{i b_i} [u_i(a) - u_i(b_i, a_{-i})] = 0
$$

Below is code to generate harmonic games from only the game skeleton. The harmonic measures are kept symbolic while solving the linear payoff system, and concrete positive measures can be chosen later to instantiate a particular harmonic game.

## Scope
This version generates a harmonic game taking as input only number of layer and number of actions and returns general solution that allows for symbolic measures and symbolic payoffs. One can then at the end specify the measures and get a payoff instance.

In [2]:
from sympy import *
import itertools as it
import random

In [3]:
# Make symbols: methods to dynamically create as many as needed strings and Sympy symbols
# Shift is starting index; default 0

def make_strings(N,s, shift = 0):
	my_strings = []
	for i in range(shift, N + shift):
	    tmp_st = f'{s}{i}'
	    my_strings.append(tmp_st)
	return my_strings

def make_symbols(N,s, shift = 0, **assumptions):
	my_symbols = []
	for i in range(shift, N + shift):
	    tmp_st = f'{s}{i}'
	    globals()[tmp_st] = Symbol(tmp_st, **assumptions)
	    my_symbols.append(globals()[tmp_st])
	return my_symbols

# Generate harmonic game

Generate the symbolic family of harmonic games. The only structural input is the number of actions per player.

## Set Game skeleton

In [4]:
# Game skeleton

# size of skeleton = number of players
# each entry of skeleton = number of actions of each player

skeleton = [2, 2]
skeleton

[2, 2]

## Symbolic Harmonic Measure

In [5]:
# Symbolic harmonic measures.
# Each mu_i_ai is assumed positive. Concrete positive values are chosen only when building an instance.

mu_symbols = [ make_symbols(skeleton[i], f'mu{i}', shift = 1, positive = True) for i in range(len(skeleton)) ]
mu_dofs = list(it.chain(*mu_symbols))

mu_symbols

[[mu01, mu02], [mu11, mu12]]

---

In [6]:
# N
numPlayers = len(skeleton)
players = range(numPlayers)

# -------------------------
# Number of action profiles
# -------------------------

# A
numPures = prod(skeleton)

# List of A_{-i} = for each player, number of action profiles of other players
numPuresMinus = [  int(numPures / skeleton[i]) for i in players ]

# -------------------------
# Number of payoff degrees of freedom
# -------------------------

# AN; number of payoff degrees of freedom
numPays = numPlayers * numPures

# sum_i A_i; number of harmonic measure dofs
numMeas = sum(skeleton)

In [7]:
# Pure actions: list of N lists, each with Ai elements; pure actions of each player

pures_play = [ make_strings(skeleton[i], f'a{i}', shift = 1) for i in players ]

In [8]:
# Pure profiles; cartesian product of pure actions of each player
# Returns one list with A = numPures elements; each element is a tuple of strings

pures = list(it.product(*pures_play))

In [9]:
assert len(pures) == numPures

In [10]:
# Pack symbolic mu in list of dicts

mu = [   dict(zip( pures_play[i], mu_symbols[i] )) for i in players    ]

In [11]:
def print_payoff(payoff_dict, payoff_symbol):
    for i in players:
        for a in pures:
            print( f'{payoff_symbol}_{i}{a} = {payoff_dict[i][a]}' )
        print()

# Profiles of other players

In [12]:
# Pure profiles of other players
# Make list of N lists; the list pure_minus[i] contains the pure action profiles of players other than i
# Build taking the cartesian product of pure actions of all players other than i
# The size of the list pure_minus[i] is A_{-i} = \prod_{j \neq i} A_j

pures_minus = [ list( it.product( *( pures_play[:i] + pures_play[i+1:] ) ) ) for i in players ]

In [13]:
for i in players:
    assert len(pures_minus[i]) == numPuresMinus[i]

# Make a

In [14]:
# Util to make (a_i, a_{-i}) given a_i and a_{-i} as tuple of strings (to be used as key for payoff dictionaries)
def make_pure(ai, a_minus_i, i):
    l = list(a_minus_i)
    return tuple(l[:i] + [ai] + l[i:])

In [15]:
def split_pure(a, i):
    
    ai = a[i]

    l = list(a)
    a_minus_i = tuple( l[:i] + l[i+1:] )
    return ai, a_minus_i

In [16]:
# check consistency
for i in players:
    for a in pures:
        ai, a_minus_i = split_pure(a, i)
        assert a == make_pure( ai, a_minus_i, i )

# Harmonic payoff

In [17]:
# harmonic payoff degrees of freedom; will build harmonic payoff h out of these

# to be determined harmonic payoff of player i (as many as A)

h_sym = [make_symbols(numPures, f'h{i}', shift = 1) for i in players]


In [18]:
h = [  dict(zip(pures , h_sym[i] ))  for i in players]


# System

In [19]:
# --------------------------------
# KEY: HARMONIC FUNCTION
# --------------------------------
def Hi(a, i):
    ai, a_minus_i = split_pure(a, i)
    return sum(   [   mu[i][bi] * ( h[i][a] - h[i][ make_pure(bi, a_minus_i, i) ] )    for bi in pures_play[i]   ]  )

In [20]:
# --------------------------------
# KEY: HARMONIC SYSTEM in degrees of freedom h.
# The system is linear in h over the symbolic coefficient field generated by mu.
# --------------------------------
system = [  sum( [  Hi(a,i)   for i in players ] )   for a in pures  ]

In [21]:
# one equation per pure in pures
assert len(system) == numPures

In [22]:
# Unknowns of system to solve: payoff degrees of freedom only.
# The mu symbols remain free parameters; solving for both h and mu at once would make the system bilinear.
dofs = list(it.chain(*h_sym))

In [23]:
numDofs = len(dofs)

In [24]:
# Unknowns of system to solve: harmonic weights
assert numDofs == numPays

In [25]:
# Matrix of linear system
A, b = linear_eq_to_matrix(system, *dofs)

In [26]:
sol = solve(system, dofs, dict = True)
sol = sol[0] if sol else {}

In [27]:
# Replace sol in dofs

extracted_sol = [e.subs(sol) for e in dofs]

# General solution here : symbolic payoffs and measure

In [42]:
# zip solution in dictionary with dofs
# Entries not solved by SymPy remain as free payoff parameters.
sol_dict = dict(zip(dofs, extracted_sol))
free_payoff_dofs = [d for d in dofs if sol_dict[d] == d]

sol_dict

{h01: h03 + h13*mu12/mu01 - h14*mu12/mu01,
 h02: h04 - h13*mu11/mu01 + h14*mu11/mu01,
 h03: h03,
 h04: h04,
 h11: h12 - h13*mu02/mu01 + h14*mu02/mu01,
 h12: h12,
 h13: h13,
 h14: h14}

---

# Build solution instance

In [43]:
# Choose concrete positive harmonic measures for an instance.
# The shape must match skeleton.

#mu_values = [  [1,1], [1,1], [1,1] ]
#mu_values = [  [1,1], [1,1], [1,1], [1,1] ]
#mu_values = [  [1,2], [1,3], [1,4] ]
#mu_values = [  [1,5], [1,6] ]
#mu_values = [  [1,2,3], [1,3,5]]
#mu_values = [  [1,1,1], [1,1,2]]
mu_values = [  [1,2], [1,1] ]

assert len(mu_values) == len(skeleton)
for i, Ai in enumerate(skeleton):
    assert len(mu_values[i]) == Ai
    assert all(mu_i_ai > 0 for mu_i_ai in mu_values[i])

fix_measure_dofs = dict(zip(mu_dofs, it.chain(*mu_values)))

In [44]:
fix_measure_dofs

{mu01: 1, mu02: 2, mu11: 1, mu12: 1}

## Symbolic payoffs once measure is specified

In [41]:
[v.subs(fix_measure_dofs) for v in sol_dict.values()]

[h03 + h13 - h14,
 h04 - h13 + h14,
 h03,
 h04,
 h12 - 2*h13 + 2*h14,
 h12,
 h13,
 h14]

In [30]:
# Fix remaining payoff dofs and the harmonic measures.
#fix_remaining_payoff_dofs = {d: random.randint(-10, 10) for d in free_payoff_dofs}
fix_remaining_payoff_dofs = {d: random.randint(-1, 1) for d in free_payoff_dofs}
fix_instance_dofs = {**fix_measure_dofs, **fix_remaining_payoff_dofs}

sol_dict_instance = { key : sol_dict[key].subs(fix_instance_dofs) for key in sol_dict}

In [31]:
sol_dict_instance

{h01: 2, h02: -3, h03: 0, h04: -1, h11: -5, h12: -1, h13: 1, h14: -1}

# Result: harmonic game

In [32]:
skeleton

[2, 2]

In [33]:
sol_dict_instance.values()

dict_values([2, -3, 0, -1, -5, -1, 1, -1])

In [34]:
numPures

4

# Order
This output give the flattened payoff vector; the first A numbers are the payoffs of the first player, the second A numbers are the payoffs of the second player, and so on.

For each player, the order in the 2x2x2 case is

```
1: (1, 1, 1)
2: (1, 1, 2)
3: (1, 2, 1)
4: (1, 2, 2)
5: (2, 1, 1)
6: (2, 1, 2)
7: (2, 2, 1)
8: (2, 2, 2)
```

# End